## 1. Import Modules and Data
Accroding to original paper <a href="https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf">Improving Language Understanding by Generative Pre-Training</a>, GPT is pretrained on bookcorpus dataset, described by [Zhu and Kiros et al](https://yknzhu.wixsite.com/mbweb). You can learn more from [here](https://huggingface.co/datasets/bookcorpus/bookcorpus) and download from [this branch](https://huggingface.co/datasets/bookcorpus/bookcorpus/tree/refs%2Fconvert%2Fparquet/plain_text/train).

The full BookCorpus dataset is too large, so we only use 10% of it for pre-training as examples. In `load_data`, all plain text will be tokenized and converted to token id with BPE algorithm. Finally, a data loader will be returned where each batch is token ids tensor with shape `[batch_size, max_len]`.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from dotenv import load_dotenv
import config
from data import load_data

from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration

#using ProjectConfiguration to automatically manage checkpoints
config_project = ProjectConfiguration(
    project_dir=str(config.checkpoint_dir),
    automatic_checkpoint_naming=True,
    total_limit=10
)

accelerator=Accelerator(
    project_config=config_project
)

device=accelerator.device
#correspond to DeepSpeed's config
if accelerator.distributed_type=="DEEPSPEED":
    accumulate_grad_batches=accelerator.deepspeed_config.gradient_accumulation_steps
else:
    accumulate_grad_batches=config.PretrainConfig.accumulate_grad_batches

accelerator.gradient_accumulation_steps = accumulate_grad_batches

load_dotenv()

# only use 1% BookCorpus dataset
loading_ratio = 0.01

with accelerator.main_process_first():
    tokenizer, dataloader = load_data("bookcorpus", loading_ratio=loading_ratio, num_proc=5)

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

## 2. Build Model

I've pointed out key structure differences between GPT and vanilla transformer in [README](./README.md), but I still want to highlight that the GeLU used in feed forward layers in decoder block is approximate with `tanh`.

In [ ]:
import torch
import config
from modules import GPT

model = GPT(
    vocab_size=config.vocab_size,
    max_len=config.max_len,
    hidden_size=config.hidden_size,
    num_attention_heads=config.num_attention_heads,
    num_hidden_layers=config.num_hidden_layers,
    dropout=config.dropout,
)

## 3. Pretrain Model
We have set most hyper parameters in [config.py](./config.py), two last things are explore optimizer and scheduler we used.

### 3.1 Optimizer
We won't use `AdamW` as optimizer directly. Instead, we should filter out those gain weights, e.g. weights and bias of `LayerNorm`, and biases to make sure they won't be applied weight decay, see `GPT.configure_optimizer()`.

### 3.2 Scheduler
The learning rate was increased linearly from zero over the first `warmup_steps` updates and annealed to 0 using a cosine schedule.

Let's visualize its image.


### 3.3 Train Loop
Let's recall the training objective of language model, given a sequence of tokens $ x_1, x_2, \ldots, x_T $ from data loader, the model is trained to predict each token $ x_t $ conditioned on the previous tokens $ x_1, x_2, \ldots, x_{t-1} $:
$$
\mathcal{L} = -\sum_{t=1}^{T} \log P(x_t | x_1, x_2, \ldots, x_{t-1}; \theta)
$$

where $ \theta $ represents the model parameters. 

Considering the GPU memory constraints, gradient accumulation is needed to ensure sufficient tokens in each batch, updating gradients every `accumulate_grad_batches` rounds.

In [ ]:
from tqdm.notebook import tqdm
import torch.nn as nn

def split_batch(batch):
    src, tgt = batch[:, :-1], batch[:, 1:]
    return src, tgt


def train(epoch, model, criterion, optimizer, scheduler,status,writer=None):
    model.train()
    total_loss = 0
    step = 0
    optimizer.zero_grad()

    process_bar=tqdm(
        dataloader,
        desc=f"Training Epoch {epoch}",
        disable=not accelerator.is_main_process,
    )

    for batch in process_bar:
        with accelerator.accumulate(model):
            src, tgt = split_batch(batch)

            # [batch_size, max_len - 1, vocab_size]
            outputs = model(src)

            # [batch_size * (max_len - 1), vocab_size]
            outputs = outputs.contiguous().view(-1, tokenizer.get_vocab_size())
            loss = criterion(outputs, tgt.contiguous().view(-1))

            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), config.clip)
                status.global_step += 1
                #redefine the step as the times arguments update

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()

            if writer is not None:
                writer.add_scalar("Loss/train", loss.item(), status.global_step)
                writer.add_scalar("Learning_rate",scheduler.get_last_lr()[0],status.global_step)

    avg_loss = total_loss / len(dataloader)

    return avg_loss

In [ ]:
import os
from accelerate.utils import set_seed
from torch.utils.tensorboard import SummaryWriter
from transformers import get_cosine_schedule_with_warmup

class TrainingStatus:
    def __init__(self, checkpoint_interval):
        self.current_epoch = 0
        self.global_step = 0
        self.checkpoint_interval = checkpoint_interval

    def state_dict(self):
        return {
            "current_epoch": self.current_epoch,
            "global_step": self.global_step,
            "checkpoint_interval": self.checkpoint_interval,
        }

    def load_state_dict(self, state_dict):
        self.current_epoch = state_dict["current_epoch"]
        self.global_step = state_dict["global_step"]
        self.checkpoint_interval = state_dict["checkpoint_interval"]

status = TrainingStatus(config.checkpoint_interval)
accelerator.register_for_checkpointing(status)

#pass the restore_iteration
def training_loop(restore_iteration=-1,seed:int=42,use_tensorboard=False):
    set_seed(seed)
    optimizer = model.configure_optimizers(
        lr=config.PretrainConfig.lr,
        weight_decay=config.weight_decay,
        device_type=accelerator.device.type,
    )

    criterion = nn.CrossEntropyLoss()

    model, optimizer, dataloader = accelerator.prepare(
        model, optimizer, dataloader
    )
    # compute correct total steps after dataloader is divided
    total_steps = (
        len(dataloader) * config.PretrainConfig.n_epoch //accumulate_grad_batches
    )
    # scale warmup steps
    warmup_steps = int(
        config.PretrainConfig.warmup_steps
        * loading_ratio
        // accumulate_grad_batches
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer=optimizer,
        num_training_steps=total_steps,
        num_warmup_steps=warmup_steps,
    )

    scheduler=accelerator.prepare(scheduler)

    restore_path = config.checkpoint_dir/f"checkpoints/checkpoint_{restore_iteration}"
    if restore_iteration != -1 and os.path.exists(restore_path):
        accelerator.load_state(restore_path)
        restore_epoch=status.current_epoch
    else:
        restore_epoch = 0

    writer=None
    if accelerator.is_main_process and use_tensorboard:
        writer=SummaryWriter(log_dir=config.checkpoint_dir / "logs")

    for epoch in range(restore_epoch, config.PretrainConfig.n_epoch):
        local_avg_loss = train(epoch + 1, model, criterion, optimizer, scheduler, status, writer)

        if (epoch+1) % config.checkpoint_interval == 0:
            status.current_epoch=epoch
            accelerator.save_state()
        
        if writer and accelerator.is_main_process:
            gathered_loss = accelerator.gather(torch.tensor(local_avg_loss))
            total_samples = accelerator.gather(torch.tensor(len(dataloader)))
            avg_train_loss = gathered_loss.sum() / total_samples.sum()
            writer.add_scalar('Loss/train_epoch', avg_train_loss, epoch)

    accelerator.wait_for_everyone()
    #save the model
    if accelerator.is_main_process:
        save_path=config.save_model_dir/f"gpt_pretrained.pth"

        unwrapped_model=accelerator.unwrap_model(model)
        accelerator.save(
            unwrapped_model.state_dict(),
            save_path
        )

args=(-1,42,True)
training_loop(*args)